# M1: save the 2,048-d feature vectors (m1_final_n500 / hp_fft)

**What this does:** loads the trained M1 checkpoint (`best.pt`), runs every image of the four splits through it
once, and saves the 2,048 numbers the ResNet-50 produces just before the final score. **No training.**

**Why:** the Fusion model needs M1's full description of each image, not only its final score.

**Read only:** checkpoint, config, split CSVs, caches, library. **Written:** only the new folder
`VeriStamp/features/m1_final_n500_hp_fft/` (the notebook stops if that folder already has files).

Runs on any runtime (A100, T4 or CPU). Run the sections in order.

## 1. Drive and paths

In [1]:
import os, sys, json, hashlib, shutil
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab: set PROJECT_DIR to your local copy of the VeriStamp Drive folder.")

PROJECT_DIR = Path("/content/drive/MyDrive/VeriStamp")      # <<< CHANGE only if your Drive folder differs
RUN_NAME, MODE = "m1_final_n500_unseen_adm", "hp_fft"                    # hp_fft = M1 (the RGB baseline is not needed)
RUN_DIR = PROJECT_DIR / "experiments" / RUN_NAME / MODE
LIB_DIR = PROJECT_DIR / "vslib_m1_final"                      # the exact code the final run used
OUT_DIR = PROJECT_DIR / "features" / f"{RUN_NAME}_{MODE}"     # NEW folder; nothing existing is written
SPLITS = ["train", "val", "test_seen", "test_unseen"]

for p in [RUN_DIR / "best.pt", RUN_DIR / "config.yaml", LIB_DIR]:
    print(("found   " if p.exists() else "MISSING ") + str(p))
    assert p.exists(), f"{p} not found"
if OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise FileExistsError(f"{OUT_DIR} already has files. This notebook never overwrites; "
                          "delete that folder yourself (it only holds features) or change OUT_DIR.")
print("Output folder (new):", OUT_DIR)

Mounted at /content/drive
found   /content/drive/MyDrive/VeriStamp/experiments/m1_final_n500_unseen_adm/hp_fft/best.pt
found   /content/drive/MyDrive/VeriStamp/experiments/m1_final_n500_unseen_adm/hp_fft/config.yaml
found   /content/drive/MyDrive/VeriStamp/vslib_m1_final
Output folder (new): /content/drive/MyDrive/VeriStamp/features/m1_final_n500_unseen_adm_hp_fft


## 2. Library, config, split and caches
The split is loaded with its fingerprint check, and every cache is checked against its split CSV:
the fingerprint stored next to the cache is a hash of the ordered `image_id` list, so a match proves
**cache row i = CSV row i**.

In [2]:
sys.path.insert(0, str(LIB_DIR))
import numpy as np, pandas as pd, torch
import vs_common, vs_genimage, vs_preprocess, vs_models, vs_native, vs_metrics

CFG = vs_common.load_yaml(RUN_DIR / "config.yaml")           # the config saved by the training run
PCFG, SEED = CFG["preprocess"], CFG["seed"]
SPLIT_NAME = CFG["data"]["split_name"]
SPLIT_DIR = PROJECT_DIR / "data" / "splits" / SPLIT_NAME
CACHE_DRIVE = PROJECT_DIR / "cache" / SPLIT_NAME
assert CFG["model"]["input_mode"] == MODE and PCFG["mode"] == "native"
print("Split:", SPLIT_NAME)
print("Preprocessing:", PCFG)

MANIFEST, FRAMES, SPLIT_INFO = vs_genimage.load_split(SPLIT_DIR)        # raises if a CSV was edited
assert SPLIT_INFO["fingerprints"] == CFG["split_fingerprints"], "not the split the model was trained on"

CACHE_LOCAL = (Path("/content/cache_features") if Path("/content").exists() else Path("cache_features")) / SPLIT_NAME
CACHE_LOCAL.mkdir(parents=True, exist_ok=True)
CACHE, META = {}, {}
for s in SPLITS:
    name = f"{s}_native{PCFG['tile_size']}"
    META[s] = json.loads((CACHE_DRIVE / f"{name}.json").read_text())
    assert META[s]["pcfg"] == PCFG and META[s]["seed"] == SEED, f"{s}: cache built with other preprocessing"
    fp = vs_preprocess._cache_fingerprint(FRAMES[s]["image_id"].tolist(), PCFG, SEED)
    assert META[s]["fingerprint"] == fp and META[s]["n"] == len(FRAMES[s]), f"{s}: cache rows do not match {s}.csv"
    CACHE[s] = CACHE_LOCAL / f"{name}.npy"
    if not CACHE[s].exists():
        shutil.copy(CACHE_DRIVE / f"{name}.npy", CACHE[s])  # local copy = faster reads; the Drive file is only read
    print(f"{s:>11}: {len(FRAMES[s]):>5} rows | cache row i = {s}.csv row i: PASS")

Split: genimage_seen-VQDM-stable_diffusion_v_1_4_unseen-ADM_min224_n500_seed42__m1_final_n500_unseen_adm
Preprocessing: {'crop_size': 224, 'jpeg_quality_range': [75, 95], 'train_hflip': True, 'train_rescale_range': None, 'min_side': 224, 'mode': 'native', 'tile_size': 256, 'align': 8, 'jpeg_subsampling': '4:2:0', 'legacy_resize_short_side': 256}
      train:  1400 rows | cache row i = train.csv row i: PASS
        val:   300 rows | cache row i = val.csv row i: PASS
  test_seen:   300 rows | cache row i = test_seen.csv row i: PASS
test_unseen:  1000 rows | cache row i = test_unseen.csv row i: PASS


## 3. Rebuild M1 and load `best.pt`

In [3]:
DEVICE = vs_common.get_device()
cfg_load = {**CFG, "model": {**CFG["model"], "pretrained": False}}   # weights come from best.pt: skip the ImageNet download
model = vs_models.build_detector(MODE, cfg_load).to(DEVICE)
ck = vs_common.load_checkpoint(RUN_DIR / "best.pt", DEVICE)
model.load_state_dict(ck["model"])                                    # strict: every weight must match
model.eval()
CKPT_SHA256 = hashlib.sha256((RUN_DIR / "best.pt").read_bytes()).hexdigest()
print(f"Device: {DEVICE} | best epoch {ck['epoch']} | val AUROC during training {ck['val_metrics']['auroc']:.4f}")
print("best.pt SHA-256:", CKPT_SHA256)

Device: cuda | best epoch 9 | val AUROC during training 0.9296
best.pt SHA-256: cc5d70720054fb5501c93e6f4cced67a361110043350abad6e3cb5c84e88615e


## 4. Extract one 2,048-d vector per image
Every split, **train included**, uses the deterministic val/test crop (224x224 at the centre of the valid
region, offset a multiple of 8). No flip, no random crop. Full float32 (no mixed precision), so the
numbers are exact and repeatable. The vector is the pooled ResNet-50 output, i.e. the input of the final layer.

In [4]:
BS, NW = (64, 2) if DEVICE.type == "cuda" else (8, 0)
FEATS, ROWS = {}, {}
for s in SPLITS:
    ds = vs_native.NativeCachedDataset(FRAMES[s], CACHE[s], PCFG["crop_size"], False, False,
                                       META[s]["valid_h"], META[s]["valid_w"], PCFG["align"])
    loader = vs_preprocess.make_loader(ds, BS, False, NW, SEED)
    feats, logits, idx = [], [], []
    with torch.no_grad():
        for b in loader:
            out = model(b["x"].to(DEVICE))
            # the final score must come from exactly these features
            assert torch.allclose(model.head(out["features"]).squeeze(1), out["logits"], atol=1e-4)
            feats.append(out["features"].cpu().numpy())
            logits.append(out["logits"].cpu().numpy())
            idx.append(b["idx"].numpy())
    idx = np.concatenate(idx)
    assert (idx == np.arange(len(ds))).all(), "loader order changed"
    lg = np.concatenate(logits).astype(np.float64)
    FEATS[s] = np.concatenate(feats).astype(np.float32)
    ROWS[s] = pd.DataFrame({"cache_index": idx, "image_id": FRAMES[s]["image_id"].to_numpy(),
                            "generator": FRAMES[s]["generator"].to_numpy(),
                            "label": FRAMES[s]["label"].astype(int).to_numpy(),
                            "logit": lg, "p_ai": 1.0 / (1.0 + np.exp(-lg))})
    print(f"{s:>11}: features {FEATS[s].shape}")

      train: features (1400, 2048)
        val: features (300, 2048)
  test_seen: features (300, 2048)
test_unseen: features (1000, 2048)


## 5. Check against the run's saved predictions, then save
If the features are right, the scores computed from them must reproduce the saved `predictions_*.csv`
(same image order, same AUROC). Small differences are expected: training-time scores used bfloat16,
here everything is float32. Nothing is saved if a check fails.

In [5]:
checks = []
for s in SPLITS:
    r = ROWS[s]
    auc = vs_metrics.classification_metrics(r["label"], r["p_ai"])["auroc"]
    saved = RUN_DIR / f"predictions_{s}.csv"
    if not saved.exists():
        checks.append({"split": s, "n": len(r), "auroc_now": auc, "result": "no saved predictions to compare"})
        continue
    old = pd.read_csv(saved).sort_values("cache_index").reset_index(drop=True)
    same_order = len(old) == len(r) and (old["image_id"].astype(str).to_numpy() == r["image_id"].astype(str).to_numpy()).all()
    corr = float(np.corrcoef(old["logit"], r["logit"])[0, 1])
    agree = float(((old["p_ai"] >= 0.5).to_numpy() == (r["p_ai"] >= 0.5).to_numpy()).mean())
    old_auc = vs_metrics.classification_metrics(old["label"], old["p_ai"])["auroc"]
    ok = same_order and corr > 0.995 and abs(auc - old_auc) < 0.005
    checks.append({"split": s, "n": len(r), "auroc_now": auc, "auroc_saved": old_auc, "logit_corr": corr,
                   "pred_agreement": agree, "same_image_order": same_order, "result": "PASS" if ok else "FAIL"})
CHECKS = pd.DataFrame(checks)
print(CHECKS.round(4).to_string(index=False))
assert (CHECKS["result"] != "FAIL").all(), "Features do not reproduce the saved predictions: nothing was saved."

OUT_DIR.mkdir(parents=True, exist_ok=True)
for s in SPLITS:
    np.save(OUT_DIR / f"{s}_features.npy", FEATS[s])
    ROWS[s].to_csv(OUT_DIR / f"{s}_index.csv", index=False)
CHECKS.to_csv(OUT_DIR / "feature_checks.csv", index=False)
INFO = {"created": vs_common.now(), "run": RUN_NAME, "model": MODE, "checkpoint": str(RUN_DIR / "best.pt"),
        "checkpoint_sha256": CKPT_SHA256, "checkpoint_epoch": ck["epoch"], "split_name": SPLIT_NAME,
        "split_fingerprints": SPLIT_INFO["fingerprints"], "preprocess": PCFG, "seed": SEED,
        "feature": "pooled ResNet-50 output (input of the final layer), float32, 2048 per image",
        "crop": "deterministic centre crop of the valid region (offset multiple of 8) for ALL splits; no augmentation",
        "precision": "float32 (no mixed precision)", "device": str(DEVICE), "torch": torch.__version__,
        "shapes": {s: list(FEATS[s].shape) for s in SPLITS},
        "row_order": "row i of <split>_features.npy = row i of <split>_index.csv = row i of <split>.csv = cache row i"}
vs_common.save_json(INFO, OUT_DIR / "features_info.json")
(OUT_DIR / "README.md").write_text(
    f"# M1 features: {RUN_NAME} / {MODE}\n\n"
    "- `<split>_features.npy`: float32 [N, 2048], one row per image.\n"
    "- `<split>_index.csv`: cache_index, image_id, generator, label, logit, p_ai for the same rows.\n"
    "- Row i = row i of the split CSV = cache row i.\n"
    "- Train features use the deterministic eval crop (no augmentation).\n"
    "- Tune Fusion on val only; test_unseen (Stable Diffusion v1.4) is for final reporting.\n"
    "- Details: `features_info.json`; checks: `feature_checks.csv`.\n")
print("Saved to", OUT_DIR)

      split    n  auroc_now                          result  auroc_saved  logit_corr  pred_agreement same_image_order
      train 1400     0.9999 no saved predictions to compare          NaN         NaN             NaN              NaN
        val  300     0.9290                            PASS       0.9296      0.9997           1.000             True
  test_seen  300     0.9191                            PASS       0.9188      0.9997           0.990             True
test_unseen 1000     0.8655                            PASS       0.8656      0.9996           0.996             True
Saved to /content/drive/MyDrive/VeriStamp/features/m1_final_n500_unseen_adm_hp_fft


## 6. Test: reload what was saved

In [6]:
tests = []
for s in SPLITS:
    F = np.load(OUT_DIR / f"{s}_features.npy")
    idx = pd.read_csv(OUT_DIR / f"{s}_index.csv")
    split_csv = pd.read_csv(SPLIT_DIR / f"{s}.csv")
    tests.append({"split": s, "shape": F.shape, "dtype": str(F.dtype),
                  "2048 columns": F.shape[1] == 2048,
                  "rows = split CSV": len(F) == len(idx) == len(split_csv),
                  "same image order": (idx["image_id"].astype(str).to_numpy() == split_csv["image_id"].astype(str).to_numpy()).all(),
                  "no NaN/inf": bool(np.isfinite(F).all()),
                  "not all zero": bool((np.abs(F).sum(1) > 0).all()),
                  "identical to memory": bool(np.array_equal(F, FEATS[s]))})
T = pd.DataFrame(tests)
print(T.to_string(index=False))
assert T.drop(columns=["split", "shape", "dtype"]).all().all(), "a test failed"
print("\nFiles:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size / 1e6:7.2f} MB")
print("\nALL FEATURE TESTS PASSED")

      split        shape   dtype  2048 columns  rows = split CSV  same image order  no NaN/inf  not all zero  identical to memory
      train (1400, 2048) float32          True              True              True        True          True                 True
        val  (300, 2048) float32          True              True              True        True          True                 True
  test_seen  (300, 2048) float32          True              True              True        True          True                 True
test_unseen (1000, 2048) float32          True              True              True        True          True                 True

Files:
  README.md                       0.00 MB
  feature_checks.csv              0.00 MB
  features_info.json              0.00 MB
  test_seen_features.npy          2.46 MB
  test_seen_index.csv             0.04 MB
  test_unseen_features.npy        8.19 MB
  test_unseen_index.csv           0.10 MB
  train_features.npy             11.47 MB
  trai